In [ ]:
import importlib.util
import os

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["PYTHONHASHSEED"] = "42"

target_path = os.path.abspath(
    os.path.join(os.getcwd(), "..", "performance.py")
)

spec = importlib.util.spec_from_file_location("performance", target_path)
performance = importlib.util.module_from_spec(spec)
spec.loader.exec_module(performance)

globals().update({k: v for k, v in performance.__dict__.items() if not k.startswith("__")})

In [ ]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

In [ ]:
data = Path("DataSets/logatec")
results_path = Path("Benchmarking-results")
random_seed = 42
n_splits = 5
test_size = 0.20
subsets = ["spring", "winter"]

Path(results_path).mkdir(parents=True, exist_ok=True)

In [ ]:
start_resource_monitor()

# Praparation
# We assume the dataset has been downloaded and unzipped manually

import json
import re

def load_raw_data(path: Path) -> pd.DataFrame:
    with open(path, mode="r") as fp:
        data = json.load(fp)

    df = []

    for position, measurements in data.items():
        digits = re.findall(r"\d+", position)
        location = tuple(int(i) for i in digits)

        # Winter dataset has measurements only in the middle (3rd) row.
        if len(location) == 1:
            location = (3, *location)

        assert len(location) == 2, f"location identifier is not length 2: {location}"

        pos_x, pos_y = location

        for device_id, samples in measurements.items():
            device_id = int(device_id)
            for sample in samples:
                timestamp, value = sample["timestamp"], sample["rss"]

                item = {"pos_x": pos_x, "pos_y": pos_y, "node": device_id, "timestamp": timestamp, "value": value}
                df.append(item)

    df = pd.DataFrame(df)
    df.timestamp = pd.to_datetime(df.timestamp, unit="s", origin="unix").astype("datetime64[s]")
    df = df.astype({"pos_x": "uint8", "pos_y": "uint8", "value": "int8", "node": "uint8"})

    return df

df = [load_raw_data(data / f"{subsets[i]}_data.json") for i in range(len(subsets))]

for idx, subset in enumerate(subsets):
    dat = []
    
    # Average the sample value within a second.
    for (x, y, node, ts), subset in df[idx].groupby(by=["pos_x", "pos_y", "node", "timestamp"]):
        avg_value = subset.value.sum(min_count=1) / subset.value.count()
        item = {"pos_x": x, "pos_y": y, "node": node, "timestamp": ts, "value": avg_value}
        dat.append(item)
    
    df[idx] = pd.DataFrame(dat)
    df[idx] = df[idx].pivot(index=["timestamp", "pos_x", "pos_y"], columns=["node"], values=["value"])
    df[idx] = df[idx].reset_index(drop=False)
    
    # After pivot, column names become tuples. Fix that.
    df[idx].columns = ["".join(map(str, col)).strip().replace("value", "node") for col in df[idx].columns.values]
    
    # Fill the NaN values with some extremely low RSS value
    df[idx] = df[idx].fillna(-180)
    
    # TODO: Should this be part of prepare-feature stage?
    # Remove datetime column
    df[idx] = df[idx].drop(columns=["timestamp"])

stop_resource_monitor(results_path / f"prepare_usage.pkl")

In [ ]:
start_resource_monitor()

#Feature generation
for idx, subset in enumerate(subsets):
    # Convert discrete values to meters
    df[idx].pos_x = (df[idx].pos_x - 1) * 1.2  # meters
    df[idx].pos_y = (df[idx].pos_y - 1) * 1.2  # meters
    
    df[idx] = df[idx].rename(columns={"pos_x": "target_x", "pos_y": "target_y"})

# Find target column(s)
targets = ["target_x", "target_y"]

# X are features, y are target(s)
features, targets = [df[i].drop(targets, axis=1) for i in range(len(subsets))], [df[i][targets] for i in range(len(subsets))]

stop_resource_monitor(results_path / f"featurize_usage.pkl")

In [ ]:
start_resource_monitor()

#Split generation
from sklearn import model_selection

class PredefinedSplit(model_selection.BaseCrossValidator):
    """Simple cross-validator for predefined train-test splits."""

    def __init__(self, indices_pairs: list[tuple[np.ndarray, np.ndarray]]):
        self.idx_pairs = indices_pairs

    def get_n_splits(self, X=None, y=None, groups=None):
        """Return the number of splitting iterations in the cross-validator"""
        return len(self.idx_pairs)

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set."""
        for train_idx, test_idx in self.idx_pairs:
            yield train_idx, test_idx

groups = None

cv = [model_selection.KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_seed,
    ),
    model_selection.ShuffleSplit(
        n_splits=n_splits,
        test_size=test_size,
        random_state=random_seed,
    )
]

cv_name = ["KFold", "Random"]

split_indices = [[[] for _ in range(len(cv))] for _ in range(len(subsets))]

for idx, subset in enumerate(subsets):
    for i in range(len(cv)):
        for train_indices, test_indices in cv[i].split(features[idx], targets[idx], groups):
                split_indices[idx][i].append((train_indices, test_indices))

stop_resource_monitor(results_path / f"split_usage.pkl")

In [ ]:
start_resource_monitor()

#Train&Evaluate
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, root_mean_squared_error, r2_score

for idx, subset in enumerate(subsets):
    cv = [PredefinedSplit(split_indices[idx][i])for i in range(len(cv_name))]
    
    estimators = [
        RandomForestRegressor(random_state=42),
        KNeighborsRegressor()
    ]
    params=[
        {
            "n_estimators": [10, 50, 100, 250, 400], 
            "max_depth": [5, 10, 30, 50, 150, 200, None]
        },
        {
            "n_neighbors": [3, 5, 10], 
            "weights": ["uniform", "distance"], 
            "p": [1, 2], 
            "leaf_size": [10, 15, 30], 
            "metric": ["minkowski", "euclidean"] 
        }
    ]
    
    for split_index in range(len(cv)):
        for index in range(len(estimators)):
            print(f"{subset}-{cv_name[split_index]}-{estimators[index].__class__.__name__}")
            gs = model_selection.GridSearchCV(
                estimator = estimators[index],
                param_grid = params[index],
                n_jobs = 5,
                error_score = "raise",
                refit = "rmse",
                scoring = {"rmse": make_scorer(root_mean_squared_error, greater_is_better=False), "r_squared": make_scorer(r2_score, greater_is_better=True)},
                cv = cv[split_index],
            )


            gs.fit(features[idx], targets[idx])
            
            results_df = pd.DataFrame(gs.cv_results_)
            
            # Select key columns to display
            cols_to_show = [
                'params',
                'mean_test_rmse',
                'std_test_rmse',
                'rank_test_rmse',
                'mean_test_r_squared',
                'std_test_r_squared',
                'rank_test_r_squared',
                'mean_fit_time',
                'mean_score_time',
            ]
            
            #print(results_df[cols_to_show].to_string(index=False))
            Path(results_path).mkdir(parents=True, exist_ok=True)
            joblib.dump(gs.best_estimator_, results_path / f"Model_{estimators[index].__class__.__name__}-{cv_name[split_index]}Split-{subset}Subset.pkl") 
            joblib.dump(results_df, results_path / f"Results_{estimators[index].__class__.__name__}-{cv_name[split_index]}Split-{subset}Subset.pkl")

stop_resource_monitor(results_path / f"gridsearch_usage.pkl")

In [ ]:
start_resource_monitor()

import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
import autokeras as ak
from sklearn.model_selection import train_test_split
import keras
import time
import gc
import shutil
from sklearn.metrics import make_scorer, root_mean_squared_error, r2_score
from  multiprocess import Process

def get_best_models(num_models, auto_model):
    top_trials = auto_model.tuner.oracle.get_best_trials(num_models)
    for trial in top_trials:
        model = auto_model.tuner.load_model(trial)
        yield model, trial.hyperparameters

metrics = {"rmse": root_mean_squared_error, "r_squared": r2_score}

def run_candidate(**kwargs):
    features      = kwargs["features"]
    targets       = kwargs["targets"]
    metrics       = kwargs["metrics"]
    subset        = kwargs["subset"]
    cv_name       = kwargs["cv_name"]
    split_index   = kwargs["split_index"]
    idx           = kwargs["idx"]
    random_seed   = kwargs.get("random_seed", 42)
    test_size     = kwargs.get("test_size", 0.2)
    results_path  = kwargs["results_path"]
    split_indices = kwargs["split_indices"]

    tf.config.experimental.enable_op_determinism()
    keras.utils.set_random_seed(random_seed)
    
    inputs  = [ak.Input(name="data_input")]
    outputs = [ak.RegressionHead(name="x_out"), ak.RegressionHead(name="y_out")]
    
    keras.utils.set_random_seed(random_seed)
    
    auto_model = ak.AutoModel(
        inputs       = inputs,
        outputs      = outputs,
        seed         = 42,
        max_trials   = 10,
        overwrite    = True,
        directory    = results_path,
        project_name = f"ExampleModel-{subset}-{cv_name[split_index]}"
    )

    print(f"{subset}-{cv_name[split_index]}")
    preped_features = [features[idx].to_numpy()]
    preped_targets  = np.hsplit(targets[idx].to_numpy(), 2)

    n_samples = preped_features[0].shape[0]
    indices   = np.arange(n_samples)
    
    train_indices, val_indices = train_test_split(
        indices, 
        test_size    = test_size, 
        random_state = random_seed, 
        shuffle      = True
    )

    X_train_list = [_[train_indices] for _ in preped_features]
    X_val_list   = [_[val_indices] for _ in preped_features]
    y_train_list = [_[train_indices] for _ in preped_targets]
    y_val_list   = [_[val_indices] for _ in preped_targets]



    auto_model.fit(
        X_train_list, 
        y_train_list, 
        validation_data = (X_val_list, y_val_list), 
        verbose         = 2
    )

    save_top_n = max(1, int(len(auto_model.tuner.oracle.trials) * 0.1))

    cv = [PredefinedSplit(split_indices[idx][i]) for i in range(len(cv_name))]
    
    results = []
    for idx, (model, hyperparameters) in enumerate(get_best_models(save_top_n, auto_model)):
        print(f"\nProcessing model {idx + 1} out of {save_top_n}")

        optimizer = model.optimizer
        del model

        scores        = {name: [] for name in metrics.keys()}
        train_times   = []
        predict_times = []

        for split_idx, (train_indices, test_indices) in enumerate(
            cv[split_index].split(preped_features[0], preped_targets[0])
        ):

            print(f"\tProcessing split {split_idx + 1} out of {cv[split_index].get_n_splits()}")
            
            keras.utils.set_random_seed(random_seed)
            model         = auto_model.tuner.hypermodel.build(hyperparameters)
            new_optimizer = type(optimizer).from_config(optimizer.get_config())
            model.compile(optimizer = new_optimizer, loss="mse")

            save_path  = os.path.join(results_path / f"ExampleModel-{subset}-{cv_name[split_index]}", f"model-{idx}-{split_idx}.keras")
            model.save(save_path)
            model_size = os.path.getsize(save_path)

            X_train = [x[train_indices] for x in preped_features]
            X_test  = [x[test_indices]  for x in preped_features]
            y_train = [y[train_indices] for y in preped_targets]
            y_test  = [y[test_indices]  for y in preped_targets]

            #train
            start_time = time.perf_counter()
            model.fit(
                X_train, y_train, 
                validation_data = (X_test, y_test), 
                epochs          = 1000, 
                callbacks       = [keras.callbacks.EarlyStopping(patience=10, min_delta=1e-4)],
                verbose         = 2
            )
            train_times.append(time.perf_counter() - start_time)

            # predict
            start_time   = time.perf_counter()
            y_pred       = model.predict(X_test, batch_size=32, verbose=2)
            y_pred       = np.squeeze(np.stack(y_pred, axis=0), axis=-1)
            predict_time = time.perf_counter() - start_time
            predict_times.append(predict_time)

            y_test = np.squeeze(np.stack(y_test, axis=0), axis=-1)
            for name, func in metrics.items():
                scores[name].append(func(y_test, y_pred))

            del model
            keras.backend.clear_session()
            gc.collect()
            
        results.append({
            "scores": {
                name: {
                    "mean": np.mean(arr), 
                    "std": np.std(arr)
                } 
                for name, arr in scores.items()
            },
            "params"   : hyperparameters.values,
            "fit_time" : {
                "mean": np.mean(train_times), 
                "std" : np.std(train_times)
            },
            "score_time": {
                "mean": np.mean(predict_times), 
                "std" : np.std(predict_times)
            }
        })
    print(results, flush = True)
    joblib.dump(results, results_path / f"Results_ExampleModel-{cv_name[split_index]}Split-{subset}Subset.pkl")

    shutil.rmtree(results_path / f"ExampleModel-{subset}-{cv_name[split_index]}")

for idx, subset in enumerate(subsets):
    for split_index in range(len(cv)):
        # We use multiprocess (the multiprocessing library doesn't work) to isolate each training run.
        # This resolves an issue where model training is dependant on the order in which the models are trained,
        # this is due to a state persisting between runs (despite all the efforts to reset the environment).
        
        p   = Process(target=run_candidate, kwargs={
            "features"      : features,
            "targets"       : targets,
            "metrics"       : metrics,
            "subset"        : subset,
            "cv_name"       : cv_name,
            "split_index"   : split_index,
            "idx"           : idx,
            "random_seed"   : random_seed,
            "test_size"     : test_size,
            "results_path"  : results_path,
            "split_indices" : split_indices
        })
        p.start()
        p.join()
    
stop_resource_monitor(results_path / f"automl_usage.pkl")